# 02 - EDA Technical Audit & Stability

Stage 1 notebook with automated output routing to Step 2 folders.


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.api import add_constant

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# ---------- Paths, Scaffolding, Config ----------
# Reproducible root: run notebook from project root directory.
# If needed, replace os.getcwd() with your explicit coursework folder path.
PROJECT_ROOT = os.getcwd()
DATA_PATH = os.path.join(PROJECT_ROOT, "online_shoppers_intention.csv")
TARGET = "Revenue"

STEP_DIRS = {
    "step1": os.path.join(PROJECT_ROOT, "step1_obtain_frame"),
    "step2": os.path.join(PROJECT_ROOT, "step2_eda"),
    "step3": os.path.join(PROJECT_ROOT, "step3_prepare"),
    "step4": os.path.join(PROJECT_ROOT, "step4_shortlist"),
    "step5": os.path.join(PROJECT_ROOT, "step5_finetune"),
    "step6": os.path.join(PROJECT_ROOT, "step6_present"),
}

for _, step_dir in STEP_DIRS.items():
    os.makedirs(os.path.join(step_dir, "plots"), exist_ok=True)
    os.makedirs(os.path.join(step_dir, "tables"), exist_ok=True)

STEP1_DIR = STEP_DIRS["step1"]
STEP1_PLOT_DIR = os.path.join(STEP1_DIR, "plots")
STEP1_TABLE_DIR = os.path.join(STEP1_DIR, "tables")

STEP2_DIR = STEP_DIRS["step2"]
STEP2_PLOT_DIR = os.path.join(STEP2_DIR, "plots")
STEP2_TABLE_DIR = os.path.join(STEP2_DIR, "tables")

STEP3_DIR = STEP_DIRS["step3"]
STEP3_PLOT_DIR = os.path.join(STEP3_DIR, "plots")
STEP3_TABLE_DIR = os.path.join(STEP3_DIR, "tables")

STEP4_DIR = STEP_DIRS["step4"]
STEP4_PLOT_DIR = os.path.join(STEP4_DIR, "plots")
STEP4_TABLE_DIR = os.path.join(STEP4_DIR, "tables")

STEP5_DIR = STEP_DIRS["step5"]
STEP5_PLOT_DIR = os.path.join(STEP5_DIR, "plots")
STEP5_TABLE_DIR = os.path.join(STEP5_DIR, "tables")

STEP6_DIR = STEP_DIRS["step6"]
STEP6_PLOT_DIR = os.path.join(STEP6_DIR, "plots")
STEP6_TABLE_DIR = os.path.join(STEP6_DIR, "tables")

BEHAVIORAL_NUMERIC_FEATURES = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
]

PLOT_DPI = 300

print("Resolved PROJECT_ROOT:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Step2 plot dir:", STEP2_PLOT_DIR)
print("Step2 table dir:", STEP2_TABLE_DIR)


In [ ]:
# ---------- Helpers ----------
def load_data(path: str) -> pd.DataFrame:
    return pd.read_csv(path)


def normalize_binary_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    binary_map = {"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0}

    for col in ["Weekend", "Revenue"]:
        if col not in out.columns:
            continue
        if pd.api.types.is_bool_dtype(out[col]):
            out[col] = out[col].astype(int)
        elif out[col].dtype == object:
            mapped = out[col].astype(str).str.strip().str.lower().map(binary_map)
            if mapped.notna().all():
                out[col] = mapped.astype(int)

    return out


def get_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    tmp = df.copy()
    for c in tmp.select_dtypes(include=["bool"]).columns:
        tmp[c] = tmp[c].astype(int)
    return tmp.select_dtypes(include=[np.number])


def save_current_figure(filename: str, plot_dir: str = STEP2_PLOT_DIR, dpi: int = PLOT_DPI) -> str:
    path = os.path.join(plot_dir, filename)
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Saved plot: {path}")
    return path


def save_table(df: pd.DataFrame, filename: str, table_dir: str = STEP2_TABLE_DIR) -> str:
    path = os.path.join(table_dir, filename)
    df.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path

## Part A - Technical Audit


In [ ]:
def plot_class_balance(df: pd.DataFrame, target: str = TARGET) -> pd.DataFrame:
    ratio = df[target].value_counts(dropna=False).rename_axis(target).reset_index(name="count")
    ratio["ratio"] = ratio["count"] / ratio["count"].sum()

    plt.figure(figsize=(7, 5))
    sns.countplot(data=df, x=target, palette="Set2")
    plt.title("Class Balance of Revenue")
    plt.xlabel(target)
    plt.ylabel("Session Count")
    plt.tight_layout()
    save_current_figure("01_class_balance_countplot.png")
    plt.show()

    save_table(ratio, "01_class_balance_ratio.csv")
    return ratio

In [ ]:
def audit_missingness(df: pd.DataFrame) -> pd.DataFrame:
    missing = pd.DataFrame({
        "feature": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_ratio": df.isna().mean().values,
    }).sort_values("missing_count", ascending=False)

    plt.figure(figsize=(12, 5))
    sns.barplot(data=missing, x="feature", y="missing_ratio", color="salmon")
    plt.title("Missing Value Ratio by Feature")
    plt.ylabel("Missing Ratio")
    plt.xlabel("Feature")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    save_current_figure("02_missing_ratio_bar.png")
    plt.show()

    miss_nonzero_cols = missing.loc[missing["missing_count"] > 0, "feature"].tolist()
    if miss_nonzero_cols:
        plt.figure(figsize=(12, 4))
        sns.heatmap(df[miss_nonzero_cols].isna(), cbar=False)
        plt.title("Missingness Pattern (Columns with Nulls)")
        plt.tight_layout()
        save_current_figure("02_missingness_heatmap.png")
        plt.show()
    else:
        print("No missing values found; skipping missingness heatmap.")

    save_table(missing, "02_missingness_summary.csv")
    return missing

In [ ]:
def plot_outliers_and_distributions(df: pd.DataFrame, features: list) -> None:
    available = [f for f in features if f in df.columns]
    if not available:
        print("No specified behavioral features found.")
        return

    for feature in available:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

        sns.boxplot(data=df, y=feature, ax=axes[0], color="#8ecae6")
        axes[0].set_title(f"Boxplot: {feature}")

        sns.histplot(data=df, x=feature, bins=40, kde=True, ax=axes[1], color="#219ebc")
        axes[1].set_title(f"Histogram: {feature}")

        plt.tight_layout()
        save_current_figure(f"03_outlier_distribution_{feature}.png")
        plt.show()

    # ANALYST NOTE (Outliers):
    # - Record features with heavy skew/extreme tails.
    # - Decide: keep, cap/winsorize, or transform.

In [ ]:
def plot_univariate_strength(df: pd.DataFrame, target: str = TARGET) -> None:
    for feature in ["PageValues", "ExitRates"]:
        if feature not in df.columns:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

        sns.boxplot(data=df, x=target, y=feature, ax=axes[0], palette="Set3")
        axes[0].set_title(f"{feature} by {target} (Boxplot)")

        sns.histplot(
            data=df,
            x=feature,
            hue=target,
            bins=40,
            stat="density",
            common_norm=False,
            kde=True,
            element="step",
            ax=axes[1],
        )
        axes[1].set_title(f"{feature} Distribution by {target}")

        plt.tight_layout()
        save_current_figure(f"04_univariate_{feature}_by_{target}.png")
        plt.show()

In [ ]:
def leakage_placeholder_table(df: pd.DataFrame, target: str = TARGET) -> pd.DataFrame:
    leakage_table = pd.DataFrame({
        "feature": [c for c in df.columns if c != target],
        "capture_timing_known_at_prediction": "TBD",
        "leakage_risk_level": "TBD",
        "auditor_comments": "",
    })
    save_table(leakage_table, "05_leakage_audit_placeholder.csv")

    # ANALYST NOTE (Leakage):
    # - Confirm each feature is available at prediction time.
    # - Mark leakage risk with rationale.
    return leakage_table

## Part C - Feature Interaction


In [ ]:
def plot_spearman_heatmap(df: pd.DataFrame) -> pd.DataFrame:
    num_df = get_numeric_df(df)
    corr = num_df.corr(method="spearman")

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Spearman Correlation Heatmap (Numeric Features)")
    plt.tight_layout()
    save_current_figure("15_spearman_correlation_heatmap.png")
    plt.show()

    save_table(corr.reset_index().rename(columns={"index": "feature"}), "15_spearman_correlation_matrix.csv")
    return corr

In [ ]:
def compute_vif(df: pd.DataFrame, target: str = TARGET) -> pd.DataFrame:
    num_df = get_numeric_df(df).copy()
    if target in num_df.columns:
        num_df = num_df.drop(columns=[target])

    num_df = num_df.dropna().copy()
    keep_cols = num_df.columns[num_df.nunique(dropna=True) > 1].tolist()
    num_df = num_df[keep_cols]

    X = add_constant(num_df, has_constant="add")
    vif_df = pd.DataFrame({
        "feature": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    })
    vif_df = vif_df[vif_df["feature"] != "const"].sort_values("VIF", ascending=False)

    save_table(vif_df, "18_vif_table.csv")
    return vif_df

## Run Stage 1


In [ ]:
df_raw = load_data(DATA_PATH)
df = normalize_binary_columns(df_raw)

print(f"Dataset shape: {df.shape}")
display(df.head())

class_balance_table = plot_class_balance(df, TARGET)
missingness_table = audit_missingness(df)
plot_outliers_and_distributions(df, BEHAVIORAL_NUMERIC_FEATURES)
plot_univariate_strength(df, TARGET)
leakage_table = leakage_placeholder_table(df, TARGET)
spearman_corr = plot_spearman_heatmap(df)
vif_table = compute_vif(df, TARGET)

display(class_balance_table)
display(missingness_table)
display(leakage_table)
display(vif_table)

print("All outputs saved to:")
print("Plots:", STEP2_PLOT_DIR)
print("Tables:", STEP2_TABLE_DIR)